# Lesson 3 · Train Your Eyes to "See" Sound — Spectrogram Intuition: Read the Picture First, Learn the FFT Later

**Learning goals**
1. Read the three axes of a spectrogram (time × frequency × energy)
2. Tell a pure tone, a chord, a frequency sweep, and noise apart just by looking at the picture
3. Build the intuition that "a spectrogram is a picture you'll be able to compute by hand once you finish L32–L53 (the audio DSP module)"

> ⚠️ This lesson **derives no formulas** — it only builds visual intuition.
> The math behind the FFT and STFT comes in L37-L45.

← **Previous lesson** · [L02 · Sampling and your first sine wave](L02_sound_digital.en.ipynb)

> Last lesson covered **sampling and digital sound**: sample rate, the time axis, and your first audible sine wave.
> This lesson is about **spectrogram intuition** — learn to read a time–frequency plot first, then worry about how to compute one.

## 1. What is a spectrogram?

Ever heard a band play live? Standing in the room, you can pick out the drums, the guitar, and the vocals — even though they're all sounding at once. Your ears are doing something in real time: **splitting the sound up by frequency and telling you how "loud" each frequency is**.

A spectrogram does exactly the same thing — it just draws the result:

```
Vertical ↑  Frequency (Hz)    high ─── violin high notes, sibilants
│
│       ████ ← bright = lots of energy at this frequency
│
│       ░░░░ ← dark = almost nothing at this frequency
│
low ─── kick drum, fundamental of the voice
└────────────────────→ Time (s)
```

What the three axes mean:
- **Horizontal axis**: time — left to right is the sound unfolding
- **Vertical axis**: frequency — from low (drums) to high (whistling)
- **Color/brightness**: energy — the brighter it is, the louder that frequency is at that moment

Once you can read a spectrogram, you have a pair of eyes that can "see" sound. The FFT (L37) and STFT (L43) later on are the mathematical machines that compute spectrograms — but read the picture first, and the math will actually mean something.

> **This lesson's telescope**: below we'll use library functions like `plt.specgram` to **draw the plots** — just treat them as a telescope. How the lenses inside are ground and assembled is something we'll take apart and build ourselves from L43 on.

**Close your eyes and recite the three axes (a checklist, not new material)**:
1. Which time slice is this column?
2. Which frequency does this row correspond to?
3. Does this bright patch mean high energy or low energy?

## Before we start: three-axis checklist (the same set as §1, for reciting with your eyes closed)

Exactly the same axes as the three questions at the end of the previous section — this is just a **self-check card**, no new material:

- Time axis: the horizontal axis is the time in seconds for each sample
- Frequency axis: the vertical axis is Hz, higher means further up
- Energy axis: the brighter the color, the more energy

L32 will redraw these three axes starting from raw samples (not a ready-made spectrogram). If you can't answer the three questions, flip back to the ASCII picture in §1.

### Two little words to meet before we start: bin and hop (they show up in the next cell's comments)

The comments in the next code cell use two words that appear for the first time in this course. You didn't skip a lesson — these are two concepts that L43 covers properly. Here's the bare minimum, in plain words:

- **bin (frequency slot)**: the vertical axis of a spectrogram (the frequency axis) **is not continuous**. It's like a row of drawers, or the tick marks on a ruler.
  Each drawer is sample rate ÷ NFFT wide. For example, with sr=16000 and NFFT=256, each drawer is 16000 ÷ 256 = **62.5 Hz** wide.
  When a frequency comes in, it can only go into **the drawer closest to it**. So for a 440 Hz pure tone, the peak the machine reads out might be
  437.5 Hz (= 62.5 × 7, the gridline closest to 440) — you'll see this exact number with your own eyes in the check cell at the end of this lesson.
  **You didn't run anything wrong — the ruler itself has slots.**
- **hop**: a spectrogram is made by cutting a long sound into little slices, analyzing each slice on its own, and stitching them back together. How many
  samples you shift to the right before cutting the next slice is called the hop. The denser the shifts (the smaller the hop), the smoother and more continuous the picture looks along the horizontal (time) axis.

The next cell's comments also include a correction in parentheses ("the resolution is set by the hop, not by the overlap itself") — that's written for **future you, coming back to review after finishing
L43**. It's completely normal not to follow it now; just skip it.

All you need to remember from this lesson is one sentence: **the frequency axis is a ruler with slots, and the reading is off from the true frequency by at most half a slot.**

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Tool: display a spectrogram with matplotlib.pyplot.specgram
# (L43–L45 will implement the STFT algorithm behind it from scratch; for now, just look at the pictures)
#
# You don't need to understand these two parameters yet; L43 covers them in detail:
#   NFFT=256    each frame takes 256 samples for the FFT → frequency resolution = sr/NFFT = 62.5 Hz/bin
#   noverlap=128  adjacent frames overlap by 128 samples → each step hops 128 samples (hop) = 8 ms, so the display looks smoother (the resolution is set by the hop, not by the overlap itself)
def show_spec(signal, sr, title, ax, ylim=4000):
    ax.specgram(signal, Fs=sr, cmap='inferno', NFFT=256, noverlap=128)
    ax.set_ylim(0, ylim)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)

# The make_sine(duration, sr, freq) you wrote in L02 wraps exactly this np.sin line
# This lesson builds signals directly with np.sin so you can focus on "reading the picture"
sr = 16000
t  = np.linspace(0, 2, 2 * sr, endpoint=False)

# Pure tone: a 440 Hz sine wave
pure_tone = np.sin(2 * np.pi * 440 * t)

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(pure_tone, sr, 'Pure tone 440 Hz (A4)', ax)
plt.tight_layout(); plt.show()
print('Look for: one bright horizontal line at 440 Hz, running across the whole time axis.')

### Reading the picture: what a pure tone looks like

- **One bright horizontal line** = only one frequency, the whole time
- Where the line sits = the frequency (440 Hz ≈ 440 on the Y axis)
- How bright the line is = the energy (constant all the way = constant amplitude)

By the end of the audio DSP module (L32–L53), you should be able to start from the math behind this picture and derive every computation inside `plt.specgram` — that's a finish line you can actually check, not an empty promise.

In [ ]:
# Chord: three frequencies sounding at once
chord = (
    np.sin(2 * np.pi * 261.6 * t) +   # C4
    np.sin(2 * np.pi * 329.6 * t) +   # E4
    np.sin(2 * np.pi * 392.0 * t)     # G4
) / 3.0

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(chord, sr, 'Chord C-E-G (Do-Mi-Sol)', ax)
plt.tight_layout(); plt.show()
print('Look for: three bright horizontal lines, at C4=261 Hz, E4=330 Hz, G4=392 Hz.')

### ⚠️ Heads-up: the next cell has one "scary-looking" line of code — here's a step stool first

The next cell builds a sweep whose frequency "climbs steadily from 200 Hz to 3000 Hz." Your first instinct is probably:
the frequency changes over time as f(t) = f0 + (f1−f0)·t/T, so why not just plug that into the sine? —
`np.sin(2*np.pi * f(t) * t)`.

**Bottom line first: this most natural, intuitive version is wrong.** The diagonal line it draws won't stop at 3000 Hz; it shoots all the way up to
nearly **5800 Hz**, almost double the target (the cell after next shows a side-by-side comparison — you can see it for yourself and change the numbers to try).

Why? Think of driving a car:

- **Frequency** is like the **speedometer**: how fast you're turning right now;
- the stuff inside the sine's parentheses (the technical name is "phase") is like the **odometer**: how many "turns" you've made in total since you set off.

When your speed keeps changing, you **can't** compute the distance as "current speed × total time" — that bills the slow stretches at the start
at the final top speed too, so of course the distance comes out inflated. The right way is to take "the speed at that moment × that little slice of time" for each little slice
and **add them up**, bit by bit. Writing that "add up bit by bit" as math is what the word "integral form" in the comment means.

For the "steady acceleration" case, the sum happens to come out with a t²/2 in it — and you've actually seen this in high-school physics:
the displacement formula for uniform acceleration, s = v₀t + at²/2. Swap "speed" for "frequency" and "displacement" for "phase",
and you get exactly the line of code in the next cell, not one symbol more.

**This lesson doesn't ask you to derive this step.** Just take one sentence with you:
**phase is the mileage that frequency builds up bit by bit — not "current frequency × total time".** The full derivation comes before L37;
for now, copy this line as is and check in the picture that "the diagonal really does climb from 200 to 3000."

In [ ]:
# Sweep (glissando): frequency rises linearly over time
f0, f1 = 200, 3000  # sweep from 200 Hz to 3000 Hz
sweep = np.sin(2 * np.pi * (f0 * t + (f1 - f0) * t**2 / (2 * t[-1])))  # integral form, instantaneous frequency f(t)=f0+(f1-f0)t/T

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(sweep, sr, 'Sweep 200→3000 Hz (glissando)', ax)
plt.tight_layout(); plt.show()
print('Look for: a diagonal line rising from low to high frequency, like a slide in a performance.')

In [ ]:
# Seeing is believing: draw the "intuitive version" and the "course version" side by side
# Intuitive version sin(2π·f(t)·t) — the heads-up above said it's wrong; one look shows where it goes wrong
T = t[-1]
wrong_sweep = np.sin(2 * np.pi * (f0 + (f1 - f0) * t / T) * t)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
show_spec(sweep,       sr, 'Course version: line stops at 3000 Hz ✅',        axes[0], ylim=6000)
show_spec(wrong_sweep, sr, 'Intuitive version: shoots to ~5800 Hz ❌ (inflated 2×)', axes[1], ylim=6000)
plt.tight_layout(); plt.show()

# The intuitive version's actual instantaneous frequency is f0 + 2·(f1-f0)·t/T (by the end, exactly double the overshoot)
print(f'Actual frequency at the end of the intuitive version ≈ {f0 + 2 * (f1 - f0)} Hz, not {f1} Hz')
print('This is "distance ≠ current speed × total time" — phase has to be added up bit by bit.')

In [ ]:
# White noise: every frequency sounding at random, all at once
rng = np.random.default_rng(42)
noise = rng.standard_normal(len(t))

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(noise, sr, 'White noise (all frequencies, randomly spread)', ax)
plt.tight_layout(); plt.show()
print('Look for: the whole picture evenly bright — energy at every frequency, at every moment.')

### Pause: why does "rolling a die for every sample" mean "energy at every frequency"?

The code above does only one thing — each sample independently picks a random number. That's clearly random **in time**,
so how does it turn into **every frequency** lighting up in the spectrogram? There's a missing bridge here; let's build it now:

Stare at the waveform of a string of random numbers. It jumps all over the place, but it contains **every kind of fast and slow**:

- Every now and then a few dozen samples happen to creep slowly upward, then slowly back down — that's a "slow, big swell," in other words a **low-frequency component**;
- much more often, two neighboring samples just leap up and down — that's a "fast, small jitter," in other words a **high-frequency component**.

Because the die doesn't favor any rhythm (each roll has nothing to do with the one before), fast and slow components are all mixed in,
in roughly equal amounts — so every frequency drawer gets about the same share of energy, and the whole picture is evenly bright.
That's also why it's called "**white** noise": just as white light is an even mix of all colors.

While we're at it, a good question: is there anything special about `standard_normal` (the standard normal distribution) in the code?
Would a different kind of random number work? **Yes.** As long as each sample is drawn independently at random, the spectrogram looks about the same —
the next cell checks this for itself with the uniform distribution `rng.uniform`.

In [ ]:
# Check it yourself: "random = all frequencies" has little to do with which random distribution you use
# Swap the normal distribution (standard_normal) for a uniform one (uniform) — the spectrograms look almost the same
noise_uniform = rng.uniform(-1, 1, len(t))

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
show_spec(noise,         sr, 'Normal-distribution white noise (original code)', axes[0])
show_spec(noise_uniform, sr, 'Uniform-distribution white noise (a different die)', axes[1])
plt.tight_layout(); plt.show()
print('The two pictures are almost identical: as long as each sample is random and independent, favoring no rhythm, the whole spectrogram lights up.')
print('The rigorous proof of "independent random in time ⇔ everything present in frequency" has to wait until you learn the DFT in L37; for now the intuition is enough.')

In [ ]:
# Four sounds side by side
signals = [
    (pure_tone, 'Pure tone 440 Hz'),
    (chord,     'Chord C-E-G'),
    (sweep,     'Sweep 200→3000 Hz'),
    (noise,     'White noise'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (sig, title) in zip(axes.flat, signals):
    show_spec(sig, sr, title, ax)
plt.suptitle('Spectrograms of four sounds compared', fontsize=13)
plt.tight_layout(); plt.show()

## 2. The "fingerprints" of four sounds

Acousticians say every kind of sound has its own spectrogram "fingerprint." Spend three minutes with spectrograms and you'll recognize them:

| Sound type | Spectrogram feature | Analogy |
|---|---|---|
| Pure tone (single frequency) | One bright horizontal line | A laser — energy concentrated in one direction |
| Chord (several frequencies) | Several parallel horizontal lines | A rainbow — several colors stacked together |
| Sweep | One diagonal line (bottom left → top right) | A siren climbing from low to high |
| White noise | The whole picture evenly bright | A beach — every frequency sounding at random, all at once |

**Key intuition**: "brightness" in a spectrogram is energy density. A bright line means almost all the energy is concentrated at that frequency, like using a magnifying glass to focus sunlight onto one spot.

## 3. What speech spectrograms look like

Real speech is more complicated than the examples above. Three terms are coming up first — **just skim them; we'll unpack them in plain words right after**. If a term doesn't make sense, don't stop; go straight to the "three pictures."

- **Formants**: the resonant frequencies of the vocal tract; they form bright horizontal bands and decide "which vowel"
- **Fundamental frequency (F0 / pitch)**: the vibration frequency of the vocal folds; about 100-200 Hz for male voices, about 180-300 Hz for female voices
- **Voiced / unvoiced**: voiced sounds (like a, e, i) have a clear harmonic structure; unvoiced sounds (like s, f) look more like noise

This kind of spectrogram is exactly what Whisper takes as input (the Mel spectrogram version, implemented in L46-L47).

---

### Three terms — let's build the blocks in plain words first (after reading this, the three lines above will click)

You only need to remember one picture per term:

- **Fundamental frequency (F0)**: how many times per second the vocal folds open and close. It sets the pitch — when you hum and slide up and down, this is what's changing.
- **Harmonics**: when the vocal folds vibrate, besides the fundamental itself, they **simultaneously** produce a whole row of frequencies at **integer multiples** of the fundamental.
  A voice with a 130 Hz fundamental also rings at 260, 390, 520, 650 … Hz, just like a string that vibrates as a whole and also
  in halves and thirds at the same time (overtones). On a spectrogram, harmonics are **a row of evenly spaced thin horizontal lines**.
  Now reread the third bullet above and it clicks: voiced sounds (where your throat buzzes) have this neat row of thin lines;
  unvoiced sounds (air-friction sounds like s and f) don't vibrate the vocal folds, so there's no row of lines, and they look more like the white noise from earlier.
- **Formants**: your throat and mouth together act like a **shape-shifting speaker cabinet**. Change the shape of your mouth and the cabinet amplifies different
  frequency ranges — the ranges that get amplified are called formants. That line "resonant frequencies of the vocal tract" in plain words is:
  "the few frequencies this cabinet naturally loves to amplify." On a spectrogram, formants are **bright horizontal bands**: wherever the cabinet brightens a stretch of harmonics,
  that stretch lights up into a band.

**The two concepts that tangle most easily, separated in one sentence**:
harmonics are "a row of thin lines" whose positions are set by the **fundamental** (all integer multiples of it);
formants are "which regions get brightened," whose positions are set by the **shape of your mouth** (they need not be integer multiples of the fundamental — 700 Hz, for instance, is not an integer multiple of 130).
When you say different vowels, the fundamental can stay exactly the same (same pitch); what changes is where the formants are — and that's why a, e, i sound different.

### Where do the numbers in the next cell come from? (Explained up front so you don't have to guess)

- `700, 1200` (vowel a), `400, 2200` (vowel e), `300, 3000` (vowel i):
  these are **typical formant positions actually measured by phoneticians** (they vary a little from person to person).
  You don't need to memorize any of them — just one sentence: **different vowels = different combinations of formant positions**.
- The coefficients `0.5 / 0.3 / 0.2`: these just make the lows a bit louder and the highs a bit softer, mimicking the rough tendency of a real voice to have "more energy in the lows."
  Change them to any other ratio and the conclusion stays the same — feel free to try.
- **An honest simplification notice**: the previous section just said a real voiced sound is "a whole row of thin harmonic lines + bright formant bands."
  But the next cell cuts a corner — it uses only **3** sine waves and draws each band's **position** directly as a single thin line.
  So you're about to see three thin lines rather than "bands." That's not a contradiction; it's a simplification.
  Want to see something closer to the real thing? The cell after next really stacks 24 harmonics to show you "a row of thin lines + bright bands."

In [ ]:
# Simulate a "vowel sequence": approximate a / e / i with different formant combinations
def vowel_approx(t, f0, f1, f2):
    """Synthesize an approximate vowel from a fundamental + two formants."""
    return (
        0.5 * np.sin(2 * np.pi * f0 * t) +
        0.3 * np.sin(2 * np.pi * f1 * t) +
        0.2 * np.sin(2 * np.pi * f2 * t)
    )

seg_dur = 0.6
sr = 16000
t_seg = np.linspace(0, seg_dur, int(seg_dur * sr), endpoint=False)

# Simplified formant parameters (not accurate speech synthesis, for the spectrogram demo only)
vowel_a = vowel_approx(t_seg, 130,  700, 1200)
vowel_e = vowel_approx(t_seg, 130,  400, 2200)
vowel_i = vowel_approx(t_seg, 130,  300, 3000)
combined = np.concatenate([vowel_a, vowel_e, vowel_i])

fig, ax = plt.subplots(figsize=(9, 3))
show_spec(combined, sr, 'Simulated vowel sequence  a → e → i  (formants shifting)', ax, ylim=4000)
plt.tight_layout(); plt.show()
print('Look for: the high-frequency formant bands sit in different places in the three segments — that is the key to telling vowels apart.')

In [ ]:
# Harmonics vs formants: see "a row of thin lines" and "bright horizontal bands" in one picture
# Idea: the vocal folds emit a whole row of harmonics on a 130 Hz fundamental (130, 260, 390, ...),
#       and the vocal tract (throat + mouth) acts like a shape-shifting speaker cabinet that boosts the harmonics near the formants (700 and 1200 Hz).
f0_voice = 130                 # fundamental: the vocal folds open and close 130 times per second
formant_centers = [700, 1200]  # the two formant positions of the vowel "a" (typical phonetics values)

def formant_gain(freq):
    """The closer to a formant center, the more it gets boosted (bell-shaped gain curve, illustration only)."""
    return 0.15 + sum(np.exp(-((freq - fc) / 150) ** 2) for fc in formant_centers)

# Stack 24 harmonics: harmonic k has frequency k × 130 Hz, its loudness set by the cabinet's gain curve
voiced = sum(
    formant_gain(k * f0_voice) * np.sin(2 * np.pi * k * f0_voice * t_seg)
    for k in range(1, 25) if k * f0_voice < sr / 2
)

fig, ax = plt.subplots(figsize=(9, 3))
show_spec(voiced, sr, 'Simulated voiced "a" with 24 harmonics: each horizontal line = one harmonic, the two brighter patches = formants', ax)
plt.tight_layout(); plt.show()
print('Reading the picture: every horizontal line sits on an integer multiple of 130 Hz (harmonics, positions set by the fundamental);')
print('     the lines near 700 and 1200 Hz are clearly brighter and gather into two "bright bands" (formants, positions set by the mouth shape).')
print('     This is the picture behind that sentence in §3: harmonics are a row of thin lines, formants are the bands that get brightened.')

## ✏️ Exercise: predict the spectrogram

Without running the code, first describe in words what the spectrograms of the signals below look like:

```python
sig_a = np.sin(2*np.pi*1000*t)  # 1000 Hz pure tone
sig_b = sig_a * np.linspace(1, 0, len(t))  # amplitude decays linearly from 1 to 0
sig_c = np.sin(2*np.pi*500*t) + np.sin(2*np.pi*1500*t)  # sum of two frequencies
```

Hint: besides frequency (where the bright line is), also think about **how the brightness changes over time** — the energy axis is the one people forget.

Write your predictions in the Markdown cell below, then run the code to check.

In [ ]:
# ✏️ Fill in your predictions (write your answers before running this cell, then compare with the spectrograms in the next cell)
predictions = {
    "sig_a_1000hz_pure": None,      # describe the spectrogram of sig_a (1000 Hz pure tone)
    "sig_b_decaying":    None,      # describe the spectrogram of sig_b (1000 Hz, decaying)
    "sig_c_chord":       None,      # describe the spectrogram of sig_c (500 Hz + 1500 Hz, two frequencies stacked)
}

unfilled = [k for k, v in predictions.items() if v is None]
assert not unfilled, f'Fill in your predictions before moving on: {unfilled}'
print('Predictions recorded. Now run the next cell to see the real spectrograms and compare.')
for k, v in predictions.items():
    print(f'  {k}: {v}')

In [ ]:
sr = 16000
t  = np.linspace(0, 2, 2 * sr, endpoint=False)

sig_a = np.sin(2 * np.pi * 1000 * t)
sig_b = sig_a * np.linspace(1, 0, len(t))
sig_c = np.sin(2 * np.pi * 500 * t) + np.sin(2 * np.pi * 1500 * t)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, sig, title in zip(axes,
        [sig_a, sig_b, sig_c],
        ['sig_a: 1000 Hz', 'sig_b: 1000 Hz decaying', 'sig_c: 500 + 1500 Hz']):
    show_spec(sig, sr, title, ax)
plt.tight_layout(); plt.show()

### Taking the next cell's "check code" apart line by line (if you know print, variables, and loops, you can follow it)

The next cell is the only code in the whole lesson that "proves with numbers that the picture isn't lying to you." It uses a few constructs you haven't learned yet, so let's translate them into plain words line by line first:

1. `spectrum, freqs, _, _ = ax.specgram(...)`
   specgram returns 4 things at once, and we only want the first two. **The two underscores `_` are a Python convention
   meaning "I don't want these two return values, throw them away"** — `_` is just an ordinary variable name that, by convention, serves as a trash can.
2. `spectrum` is a 2-D table, **shape = (number of frequency bins, number of time frames)**; here it comes out as `(513, 17)`:
   each **row** is one frequency drawer (bin), each **column** is one time slice.
3. `spectrum.mean(axis=1)`: `axis=1` means **average along the time direction** — squash the 17 time frames into 1 column,
   giving "the average energy in each frequency drawer" (513 numbers). An easy rule to remember: **whichever axis number you pass, that dimension gets "eaten"**
   (axis=1 eats dimension 1, the time frames, leaving the frequency dimension).
   Careful: if you mistakenly write `axis=0`, it eats the frequency dimension instead, and argmax finds "the **time frame** with the most energy" —
   a completely different meaning, and the code won't raise an error. Watch out for this kind of "runs fine but means the wrong thing" trap whenever you write NumPy.
4. `.argmax()`: among these 513 numbers, find **the position (row number) of the maximum** — note it returns the position, not the maximum value itself.
5. `freqs[peak_bin]`: use that row number to look up the actual frequency in `freqs` (the frequency for each row).

And two shots of prevention up front:

- In this cell `NFFT` changes from the 256 we've used all along to **1024**. No conspiracy: the drawers get narrower
  (16000 ÷ 1024 = 15.625 Hz/bin), so the frequency is pinned down more precisely and the check is more convincing. That's all.
- The printed peak will be **437.5 Hz, not 440 Hz**. Don't panic — this is exactly the "ruler with slots" from the start of the lesson:
  440 ÷ 15.625 = 28.16, so it lands in drawer number 28, whose gridline is at 28 × 15.625 = **437.5 Hz**.
  The 2.5 Hz gap is less than half a slot wide and completely normal — the `< 20` tolerance in the `assert` is exactly the margin left for this kind of half-slot error.

In [ ]:
# Check: use the frequency axis returned by plt.specgram to confirm the 440 Hz pure tone peaks in the right bin
sr = 16000
t  = np.linspace(0, 1, sr, endpoint=False)
pure_tone = np.sin(2 * np.pi * 440 * t)

# specgram returns (spectrum, freqs, t_bins, im); `_` = "I don't want this return value"
# Note: NFFT is 1024 here (not the 256 from before) — narrower drawers = more precise frequency, see the breakdown above
fig, ax = plt.subplots(figsize=(6, 2))
spectrum, freqs, _, _ = ax.specgram(pure_tone, Fs=sr, NFFT=1024)
plt.close()

# Find the frequency bin where the peak is
# spectrum shape = (number of frequency bins, number of time frames); mean(axis=1) = average over time, squashing the time frames into one average spectrum
peak_bin = spectrum.mean(axis=1).argmax()
peak_freq = freqs[peak_bin]
print(f'Peak bin frequency: {peak_freq:.1f} Hz (expected ≈ 440 Hz)')
assert abs(peak_freq - 440) < 20, f'Peak is too far off: {peak_freq:.1f} Hz'
print('✅ The spectrogram peak of the 440 Hz pure tone lands in the right frequency bin')

# Why does it print 437.5 rather than 440? — the frequency ruler has slots (previewed in the cell above; here the numbers confirm it)
bin_width = sr / 1024
nearest_gridline = round(440 / bin_width) * bin_width
print(f'spectrum shape = {spectrum.shape} (frequency bins, time frames)')
print(f'bin width = {bin_width:.3f} Hz; gridline closest to 440 Hz = {nearest_gridline:.1f} Hz — matches the peak, the picture is not lying to you')


## Wrap-up

You can now **read** a spectrogram, but you don't yet know **how it's computed**.

| Intuition | Math | Where in the course |
|---|---|---|
| Frequency = position on the Y axis | The DFT's output frequency points | L37-L39 |
| Energy = color brightness | Squared magnitude of a complex number | L40-L41 |
| Stepping right one column at a time along the time axis ≈ cutting a short slice each time and analyzing it (this is the hop) | STFT; window functions in L36 | L36, L43-L45 |
| Mel version of the spectrogram | Mel filterbank | L46-L47 |

By the end of the audio DSP module (L32–L53), you'll be able to compute every spectrogram in this lesson from scratch —
including every step inside `plt.specgram`, all in code you wrote yourself.

**Next lesson L04**: the three ingredients of a sine wave — what frequency, amplitude, and phase each control.
Once you've got those, L06's Euler's formula gives the FFT its concrete "bricks."
---
⬇️ **Completion check**: you've read the wrap-up; please finish the "Spectrogram reading challenge" below before ticking the self-review.


## ✏️ Spectrogram reading challenge: infer the sound from the picture alone, no code

**This is the core exercise of the lesson.** Spectrograms come before formulas — you won't learn how the FFT is computed until L37, but you need to be able to read the picture now.

---

**The task**: here are text descriptions of three spectrograms; infer which type of sound each one is:

**Figure A (图A)**: the whole picture is evenly bright from top to bottom, with no especially bright horizontal or diagonal lines.
→ What sound is this? (pure tone / chord / sweep / white noise)

**Figure B (图B)**: there are three bright horizontal lines, at about 261 Hz, 330 Hz, and 392 Hz, unchanged from left to right.
→ What sound is this? Describe what it means musically.
(**Plain words count too**: you don't need terms like "C major triad" — "three steady pitches sounding at once" is fine.)

**Figure C (图C)**: there is one bright diagonal line climbing from the bottom left (200 Hz) to the top right (3000 Hz).
→ What sound is this? Which performance technique does it correspond to in music?
(**Plain words count too**: "the pitch climbs continuously / a siren going from low to high" is fine; you don't need to know the word glissando.)

**Figure D (图D)**: there is a bright horizontal line, but it gets dimmer from left to right until it almost disappears.
→ What sound is this?

---

Write your answers into the dictionary in the check cell below, and the code will tell you whether you're right.

In [ ]:
# ✏️ Fill your answers into the dictionary, then run to check
# (The keys "图A"…"图D" are Figure A…D; leave them as they are — the checker below uses them.)
import numpy as np

quiz_answers = {
    "图A": None,  # fill in 'white noise' / 'pure tone' / 'chord' / 'sweep'
    "图B": None,  # fill in 'chord' + an explanation (e.g. 'C major triad chord')
    "图C": None,  # fill in 'sweep' + an explanation
    "图D": None,  # fill in 'decaying pure tone' or a description
}

unfilled = [k for k, v in quiz_answers.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'

# Check: the answer should contain a keyword
# (The grader only does a "rough keyword check" and can't understand human language; right meaning in different words still counts — see the note at the end of the output)
correct = {
    "图A": ["白噪声", "noise", "随机", "所有频率", "全频"],
    "图B": ["和弦", "chord", "多频", "三和弦", "同时", "三个", "do", "mi", "sol", "c-e-g", "ceg"],
    "图C": ["扫频", "sweep", "glissando", "上行", "滑音", "爬升", "从低到高", "升高"],
    "图D": ["衰减", "decay", "envelope", "减弱", "变小", "变弱", "越来越", "消失", "渐弱", "变暗"],
}

all_correct = True
for fig, keywords in correct.items():
    ans = str(quiz_answers.get(fig, "")).lower()
    kw_lower = [k.lower() for k in keywords]
    if any(k in ans for k in kw_lower):
        print(f'  {fig} ✅')
    else:
        print(f'  {fig} ⚠️  compare against the four fingerprints in §2, then fill it in again; do not dig through this cell for the answer keywords')
        all_correct = False

if all_correct:
    print('\n🎉 Spectrogram reading challenge passed! You can now "hear" sound with your eyes.')
else:
    print('\n💡 Review the pattern summary in §2, compare it with the spectrogram legend, then edit your answers and run again.')
    print('📢 Honest disclaimer: this cell only does a "rough keyword check" and cannot really understand what you wrote.')
    print('   If your answer says the same thing in different words (e.g. Figure B: "three notes at once", Figure D: "the sound fades away"),')
    print('   then you got it right — confirm it yourself against the fingerprint table in §2; no need to bend your wording to suit the grader.')

In [ ]:
# ✏️ Self-review for this lesson
l03_review = {
    "three_axes_understood":   None,  # Can you name the three axes of a spectrogram (time/frequency/energy)? True/False
    "four_patterns_recognized": None, # Can you recognize the spectrogram fingerprints of the four sounds? True/False
    "prediction_exercise_done": None, # Did you fill in the prediction exercise before running it? True/False
    "quiz_passed":              None, # Did you pass the spectrogram reading challenge? True/False
}

unfilled = [k for k, v in l03_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l03_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L03 complete! On to L04: the three ingredients of a sine wave')

---

→ **Next lesson** · [L04 · The three ingredients of a sine wave](../1_complex_trig/L04_trig.en.ipynb)

> Next lesson covers **the three ingredients of a sine wave**: frequency sets the pitch, amplitude sets the loudness, phase sets the starting point — and you'll implement it yourself.